# LeetCode 3379 — Transformed Array bằng Python

Link đề: https://leetcode.com/problems/transformed-array/ · Độ khó: **Easy** · Chủ đề: Array (mảng), Simulation (mô phỏng), Modulo (phép chia lấy dư)

Skeleton LeetCode:

```python
class Solution:
    def constructTransformedArray(self, nums: List[int]) -> List[int]:
        ...
```

Ghi chú: hai ví dụ chính lấy từ đề; mọi đáp án được kiểm chứng bằng code (đối chiếu bốn cách làm với một bộ mô phỏng "đi từng bước" trên nhiều input ngẫu nhiên đúng ràng buộc). Ví dụ tự dựng thêm được ghi rõ "tự dựng" — nếu cần khớp từng chữ thì đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: `nums` là một **mảng vòng** (cuối nối đầu). Với **mỗi** chỉ số `i` (độc lập nhau), nếu `nums[i] > 0` đi sang **phải** `nums[i]` bước, nếu `< 0` đi sang **trái** `|nums[i]|` bước, nếu `== 0` đứng yên; `result[i]` là giá trị ở chỗ **đáp xuống**.
2. Nắm các công cụ Python: phép `%` (đặc biệt **với số âm**), `enumerate`, list comprehension, `abs`, biểu thức điều kiện `a if cond else b`, và khác biệt giữa **sao chép** `list(nums)` với **gán** `result = nums`.
3. Đi từ cách **đi từng bước một có kiểm tra vượt biên** → **tách hướng phải/trái với `%`** → **công thức gộp `(i + nums[i]) % n`** → **công thức an toàn kiểu C++/Java `(i + x % n + n) % n`**.
4. Hiểu mấu chốt: chỉ số đáp xuống là `(i + nums[i]) mod n` — phép `mod` của Python cho kết quả **không âm** nên xử lý luôn cả bước trái; và `result[i]` phải đọc từ **mảng gốc**, không phải mảng đang bị sửa.
5. Tránh các lỗi kinh điển: sửa `nums` tại chỗ trong lúc đọc, quên vòng (`IndexError` hoặc đọc sai khi bước lớn), bỏ qua dấu của `nums[i]`, và (khi chuyển sang C++/Java) quên rằng `%` với số âm trả số âm.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` biểu diễn một **mảng vòng**. Hãy tạo mảng mới `result` **cùng kích thước** theo luật sau. Với **mỗi** chỉ số `i` (`0 <= i < len(nums)`), thực hiện **độc lập**:

- Nếu `nums[i] > 0`: bắt đầu tại `i`, đi `nums[i]` bước về **bên phải** (trong mảng vòng); đặt `result[i]` = giá trị tại chỗ đáp xuống.
- Nếu `nums[i] < 0`: bắt đầu tại `i`, đi `abs(nums[i])` bước về **bên trái**; đặt `result[i]` = giá trị tại chỗ đáp xuống.
- Nếu `nums[i] == 0`: đặt `result[i] = nums[i]` (tức `0`).

Trả về `result`. (**Mảng vòng:** đi quá phần tử cuối thì quay về đầu mảng; đi lùi quá phần tử đầu thì quay về cuối.)

**Ví dụ 1**

- Input: `nums = [3, -2, 1, 1]`
- Output: `[1, 1, 1, 3]`
- Giải thích: `i=0` đi phải 3 bước tới `nums[3] = 1`; `i=1` đi trái 2 bước tới `nums[3] = 1`; `i=2` đi phải 1 bước tới `nums[3] = 1`; `i=3` đi phải 1 bước (quay vòng) tới `nums[0] = 3`.

**Ví dụ 2**

- Input: `nums = [-1, 4, -1]`
- Output: `[-1, -1, 4]`
- Giải thích: `i=0` đi trái 1 bước (quay vòng) tới `nums[2] = -1`; `i=1` đi phải 4 bước tới `nums[2] = -1`; `i=2` đi trái 1 bước tới `nums[1] = 4`.

**Ràng buộc**

- `1 <= nums.length <= 100`
- `-100 <= nums[i] <= 100`

In [1]:
from typing import List

examples = [
    ([3, -2, 1, 1], [1, 1, 1, 3]),
    ([-1, 4, -1], [-1, -1, 4]),
]

def transform_quick(nums):
    """Bản nhanh để chạy thử ví dụ: công thức (i + nums[i]) % n."""
    n = len(nums)
    return [nums[(i + x) % n] for i, x in enumerate(nums)]

for nums, expected in examples:
    print(f"nums={nums} -> {transform_quick(nums)} (kỳ vọng {expected})")
    assert transform_quick(nums) == expected

nums=[3, -2, 1, 1] -> [1, 1, 1, 3] (kỳ vọng [1, 1, 1, 3])
nums=[-1, 4, -1] -> [-1, -1, 4] (kỳ vọng [-1, -1, 4])


## 3. Kiến thức nền

### 3.1. Mảng vòng (circular array) và "đi từng bước"

Mảng vòng nghĩa là sau phần tử cuối là phần tử đầu, và trước phần tử đầu là phần tử cuối. Với mảng `[a, b, c, d]` (n = 4), từ chỉ số `3` đi **phải 1 bước** tới chỉ số `0`; từ chỉ số `0` đi **trái 1 bước** tới chỉ số `3`. Ta có thể mô phỏng bằng cách đi từng bước một và kiểm tra: nếu vượt biên thì "quay vòng" bằng một lệnh `if`.

In [2]:
n = 4
idx = 3
idx += 1                  # đi phải 1 bước
if idx == n:              # vượt biên phải -> về đầu
    idx = 0
print("từ 3 đi phải 1 bước ->", idx)    # 0

idx = 0
idx -= 1                  # đi trái 1 bước
if idx < 0:               # vượt biên trái -> về cuối
    idx = n - 1
print("từ 0 đi trái 1 bước ->", idx)    # 3

từ 3 đi phải 1 bước -> 0
từ 0 đi trái 1 bước -> 3


### 3.2. Phép `%` (chia lấy dư) và quay vòng

`a % n` là phần dư khi chia `a` cho `n`. Với `n` dương, `a % n` luôn nằm trong `0 .. n-1`. Vì vậy nó chính là công cụ **quay vòng**: đi quá cuối thì "về đầu" tương ứng với `% n`.

**Điểm quan trọng với số âm trong Python:** `-1 % 4 = 3`, `-5 % 4 = 3`. Kết quả **luôn không âm** (cùng dấu với số chia). Nhờ vậy `(i + x) % n` với `x` âm vẫn cho chỉ số đúng trong `0..n-1` — **không cần thêm gì**. (Trong C++/Java, `-1 % 4 = -1`, nên cần thêm `+ n`: xem Cách 4.)

In [3]:
n = 4
for a in (0, 3, 4, 5, 9, -1, -2, -4, -5, -9):
    print(f"{a:>3} % {n} = {a % n}")

  0 % 4 = 0
  3 % 4 = 3
  4 % 4 = 0
  5 % 4 = 1
  9 % 4 = 1
 -1 % 4 = 3
 -2 % 4 = 2
 -4 % 4 = 0
 -5 % 4 = 3
 -9 % 4 = 3


### 3.3. `enumerate` và list comprehension

`enumerate(nums)` trả từng cặp `(i, x)` gồm **chỉ số** và **giá trị**. List comprehension `[biểu_thức for i, x in enumerate(nums)]` dựng một **danh sách mới** từ biểu thức tính cho mỗi cặp — gọn hơn vòng `for` + `append`, và luôn tạo danh sách **mới** (không sửa danh sách cũ).

In [4]:
nums = [3, -2, 1, 1]
for i, x in enumerate(nums):
    print(i, x)

print([x * 2 for x in nums])                       # [6, -4, 2, 2]
print([(i, x) for i, x in enumerate(nums)])        # [(0, 3), (1, -2), (2, 1), (3, 1)]
print(nums)                                        # nums không bị đổi

0 3
1 -2
2 1
3 1
[6, -4, 2, 2]
[(0, 3), (1, -2), (2, 1), (3, 1)]
[3, -2, 1, 1]


### 3.4. `abs` và biểu thức điều kiện `a if cond else b`

`abs(x)` trả giá trị tuyệt đối: `abs(-2) = 2`. Biểu thức điều kiện `a if cond else b` chọn `a` nếu `cond` đúng, ngược lại `b` — hữu ích để tách ba trường hợp **dương / âm / bằng 0**.

In [5]:
for x in (3, -2, 0):
    huong = "phải" if x > 0 else ("trái" if x < 0 else "đứng yên")
    print(f"x={x:>2}: hướng {huong}, số bước = {abs(x)}")

x= 3: hướng phải, số bước = 3
x=-2: hướng trái, số bước = 2
x= 0: hướng đứng yên, số bước = 0


### 3.5. Sao chép danh sách: `result = nums` KHÁC `list(nums)`

`result = nums` **không** tạo bản sao: hai tên cùng trỏ vào **một** danh sách, sửa cái này là đổi cái kia. `list(nums)` (hoặc `nums[:]`, `nums.copy()`) mới tạo **bản sao** độc lập. Điều này quan trọng vì đề yêu cầu mỗi `result[i]` đọc từ mảng **gốc**; nếu vừa đọc vừa sửa cùng một danh sách, các phần tử sau sẽ đọc phải giá trị **đã bị đổi**.

In [6]:
nums = [1, 2, 3]
alias = nums                 # cùng một danh sách
copy = list(nums)            # bản sao độc lập
alias[0] = 99
print("nums :", nums)        # [99, 2, 3]  (bị đổi theo alias)
print("copy :", copy)        # [1, 2, 3]   (không đổi)
print(alias is nums, copy is nums)   # True False

nums : [99, 2, 3]
copy : [1, 2, 3]
True False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `nums = [3, -2, 1, 1]` (`n = 4`), vẽ các chỉ số `0 1 2 3` trên một vòng tròn.

| i | `nums[i]` | hướng | đi bao nhiêu bước | chỉ số đáp xuống | `result[i]` |
|---|---|---|---|---|---|
| 0 | 3 | phải | 3 | `0 → 1 → 2 → 3` = 3 | `nums[3] = 1` |
| 1 | -2 | trái | 2 | `1 → 0 → 3` = 3 | `nums[3] = 1` |
| 2 | 1 | phải | 1 | `2 → 3` = 3 | `nums[3] = 1` |
| 3 | 1 | phải | 1 | `3 → 0` (quay vòng) = 0 | `nums[0] = 3` |

Kết quả `[1, 1, 1, 3]`. Quan sát: chỉ số đáp xuống luôn là **`i + nums[i]`** (cộng nếu dương, **trừ `|nums[i]|`** nếu âm — tức vẫn là `i + nums[i]` vì số âm cộng vào là trừ), rồi **quay vòng**. Với `i = 1`: `1 + (-2) = -1`, chỉ số `-1` trên vòng tròn có `4` vị trí chính là `3`. Với `i = 3`: `3 + 1 = 4` ≡ `0`.

Trường hợp `nums[i] = 0`: đứng yên ⇒ `result[i] = nums[i] = 0`; công thức `i + 0 = i` cũng cho `nums[i]` — **không cần xử lý riêng**.

In [7]:
def show_transform(nums):
    n = len(nums)
    print(f"nums = {nums}, n = {n}")
    for i, x in enumerate(nums):
        raw = i + x                      # có thể ngoài 0..n-1
        land = raw % n                   # quay vòng
        print(f"  i={i}: nums[i]={x:>3}  i+nums[i]={raw:>3}  ->  chỉ số {land}  ->  result[{i}] = nums[{land}] = {nums[land]}")

show_transform([3, -2, 1, 1])
show_transform([-1, 4, -1])

nums = [3, -2, 1, 1], n = 4
  i=0: nums[i]=  3  i+nums[i]=  3  ->  chỉ số 3  ->  result[0] = nums[3] = 1
  i=1: nums[i]= -2  i+nums[i]= -1  ->  chỉ số 3  ->  result[1] = nums[3] = 1
  i=2: nums[i]=  1  i+nums[i]=  3  ->  chỉ số 3  ->  result[2] = nums[3] = 1
  i=3: nums[i]=  1  i+nums[i]=  4  ->  chỉ số 0  ->  result[3] = nums[0] = 3
nums = [-1, 4, -1], n = 3
  i=0: nums[i]= -1  i+nums[i]= -1  ->  chỉ số 2  ->  result[0] = nums[2] = -1
  i=1: nums[i]=  4  i+nums[i]=  5  ->  chỉ số 2  ->  result[1] = nums[2] = -1
  i=2: nums[i]= -1  i+nums[i]=  1  ->  chỉ số 1  ->  result[2] = nums[1] = 4


## 5. Cách 1 — Brute Force / baseline: đi từng bước một, kiểm tra vượt biên

**Ý tưởng.** Làm đúng như đề, **mô phỏng thật**: với mỗi `i`, đặt `pos = i`. Nếu `nums[i] > 0` thì lặp `nums[i]` lần: `pos += 1` và nếu `pos == n` thì `pos = 0`. Nếu `nums[i] < 0` thì lặp `|nums[i]|` lần: `pos -= 1`, và nếu `pos < 0` thì `pos = n - 1`. Cuối cùng `result.append(nums[pos])`. (Với `nums[i] == 0`, không lặp lần nào, `pos = i`, nên `result[i] = nums[i]`.)

In [8]:
def transform_walk(nums: List[int]) -> List[int]:
    n = len(nums)
    result = []
    for i in range(n):
        pos = i
        if nums[i] > 0:
            for _ in range(nums[i]):         # đi phải từng bước
                pos += 1
                if pos == n:
                    pos = 0
        elif nums[i] < 0:
            for _ in range(-nums[i]):        # đi trái từng bước
                pos -= 1
                if pos < 0:
                    pos = n - 1
        result.append(nums[pos])
    return result

for nums, expected in examples:
    print(transform_walk(nums), "(kỳ vọng", expected, ")")

[1, 1, 1, 3] (kỳ vọng [1, 1, 1, 3] )
[-1, -1, 4] (kỳ vọng [-1, -1, 4] )


**Độ phức tạp (Cách 1).** Mỗi `i` đi tối đa `|nums[i]| <= 100` bước ⇒ **Time `O(n * M)`** với `M = max|nums[i]| = 100`, tức `~10^4` bước. **Space `O(n)`** cho `result`.

**Nút thắt.** Mỗi bước chỉ để **quay vòng**; ta có thể tính thẳng vị trí cuối bằng một phép `%`, **không cần đi từng bước**. Cách 1 vẫn đúng và hữu ích làm **oracle** để kiểm tra các công thức ở dưới.

## 6. Ý tưởng thuật toán / bước đột phá

**Chỉ số đáp xuống = `(i + nums[i]) mod n`.** Đi `x` bước sang phải/trái trên vòng tròn `n` vị trí chỉ cần cộng `x` (âm nếu đi trái) vào `i` rồi **lấy dư** cho `n`. Không cần chia ba trường hợp: khi `nums[i] = 0` thì `(i + 0) % n = i`, đúng như đề.

Vì phép `%` của Python trả **không âm** cho số chia dương, ta không cần xử lý âm riêng. Mỗi `result[i] = nums[(i + nums[i]) % n]` **chỉ đọc** mảng gốc ⇒ có thể dùng list comprehension (mảng mới, không sửa `nums`).

In [9]:
# Vài phép tính minh hoạ cho nums = [3, -2, 1, 1] (n = 4)
n = 4
print((0 + 3) % n)       # 3
print((1 + -2) % n)      # 3   (-1 % 4 = 3: quay vòng sang trái)
print((2 + 1) % n)       # 3
print((3 + 1) % n)       # 0   (4 % 4 = 0: quay vòng sang phải)

# Bước rất lớn cũng được: nums = [100] (n = 1): (0 + 100) % 1 = 0
print((0 + 100) % 1)     # 0
# n = 3, đi trái 7 bước từ i = 1: (1 - 7) % 3 = -6 % 3 = 0
print((1 - 7) % 3)       # 0

3
3
3
0
0
0


## 7. Các cách tiếp theo

### Cách 2 — Tách hai hướng, dùng `%` cho từng hướng

**Giải thích từng đoạn.** Giữ cấu trúc "ba trường hợp" của đề nhưng thay vòng lặp bằng `%`:

1. `x > 0`: đáp xuống `(i + x) % n`.
2. `x < 0`: đi trái `-x` bước: `(i - (-x)) % n = (i + x) % n` — viết tường minh `(i - abs(x)) % n` cho dễ hiểu hướng đi.
3. `x == 0`: đáp xuống `i`, `result[i] = nums[i] = 0`.

In [10]:
def transform_branch(nums: List[int]) -> List[int]:
    n = len(nums)
    result = []
    for i in range(n):
        x = nums[i]
        if x > 0:
            land = (i + x) % n            # đi phải x bước
        elif x < 0:
            land = (i - abs(x)) % n       # đi trái |x| bước
        else:
            land = i                      # đứng yên
        result.append(nums[land])
    return result

for nums, expected in examples:
    print(transform_branch(nums), "(kỳ vọng", expected, ")")

[1, 1, 1, 3] (kỳ vọng [1, 1, 1, 3] )
[-1, -1, 4] (kỳ vọng [-1, -1, 4] )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
from typing import List

class SolutionBranch:
    def constructTransformedArray(self, nums: List[int]) -> List[int]:
        n = len(nums)
        result = []
        for i in range(n):
            x = nums[i]
            if x > 0:
                land = (i + x) % n
            elif x < 0:
                land = (i - abs(x)) % n
            else:
                land = i
            result.append(nums[land])
        return result

print(SolutionBranch().constructTransformedArray([-1, 4, -1]))   # [-1, -1, 4]

[-1, -1, 4]


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(n)`** (mảng kết quả).

### Cách 3 — Công thức gộp `(i + nums[i]) % n` bằng list comprehension

**Giải thích.** Nhận ra mọi nhánh đều là `(i + x) % n` (kể cả `x = 0`): ta có `result = [nums[(i + x) % n] for i, x in enumerate(nums)]`. Đây là dạng ngắn gọn nhất trong Python và là một biến thể của lời giải chính thức.

In [12]:
def transform_formula(nums: List[int]) -> List[int]:
    n = len(nums)
    return [nums[(i + x) % n] for i, x in enumerate(nums)]

for nums, expected in examples:
    print(transform_formula(nums), "(kỳ vọng", expected, ")")

[1, 1, 1, 3] (kỳ vọng [1, 1, 1, 3] )
[-1, -1, 4] (kỳ vọng [-1, -1, 4] )


In [13]:
# Code nộp LeetCode cho Cách 3
from typing import List

class SolutionFormula:
    def constructTransformedArray(self, nums: List[int]) -> List[int]:
        n = len(nums)
        return [nums[(i + x) % n] for i, x in enumerate(nums)]

print(SolutionFormula().constructTransformedArray([3, -2, 1, 1]))   # [1, 1, 1, 3]

[1, 1, 1, 3]


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`**.

### Cách 4 — Công thức an toàn kiểu C++/Java: `(i + x % n + n) % n`

**Giải thích.** Trong C++/Java, `%` **giữ dấu của số bị chia**: `-1 % 4 = -1` (không phải `3`). Nên `(i + x) % n` có thể ra chỉ số âm. Cách khắc phục chuẩn:

1. `x % n` thu nhỏ bước về khoảng `(-n, n)` (có thể âm nếu `x` âm).
2. `i + x % n` có thể nằm trong `(-n, 2n)`.
3. `+ n` đẩy về dương: `(i + x % n + n)` nằm trong `(0, 3n)`.
4. `% n` cuối cùng đưa về `0..n-1`.

Trong **Python**, `x % n` đã không âm nên công thức này vẫn đúng (chỉ thừa `+ n`). Dưới đây ta **mô phỏng kiểu C** bằng hàm `c_mod` để chứng tỏ công thức hoạt động ngay cả khi `%` có thể âm.

In [14]:
def c_mod(a: int, b: int) -> int:
    """Mô phỏng phép % của C/Java: kết quả cùng dấu với a (cắt về 0)."""
    q = abs(a) // abs(b)
    q = q if (a >= 0) == (b >= 0) else -q
    return a - b * q

for a, b in [(7, 4), (-7, 4), (-1, 4), (-8, 4), (5, 3), (-5, 3)]:
    print(f"C: {a:>3} % {b} = {c_mod(a, b):>2}    Python: {a:>3} % {b} = {a % b}")

C:   7 % 4 =  3    Python:   7 % 4 = 3
C:  -7 % 4 = -3    Python:  -7 % 4 = 1
C:  -1 % 4 = -1    Python:  -1 % 4 = 3
C:  -8 % 4 =  0    Python:  -8 % 4 = 0
C:   5 % 3 =  2    Python:   5 % 3 = 2
C:  -5 % 3 = -2    Python:  -5 % 3 = 1


In [15]:
def transform_cstyle(nums: List[int]) -> List[int]:
    n = len(nums)
    return [nums[(i + c_mod(x, n) + n) % n] for i, x in enumerate(nums)]

for nums, expected in examples:
    print(transform_cstyle(nums), "(kỳ vọng", expected, ")")

[1, 1, 1, 3] (kỳ vọng [1, 1, 1, 3] )
[-1, -1, 4] (kỳ vọng [-1, -1, 4] )


In [16]:
# Code nộp LeetCode cho Cách 4 (viết trực tiếp bằng Python; công thức này cũng chuyển thẳng sang C++/Java)
from typing import List

class SolutionSafe:
    def constructTransformedArray(self, nums: List[int]) -> List[int]:
        n = len(nums)
        return [nums[(i + nums[i] % n + n) % n] for i in range(n)]

print(SolutionSafe().constructTransformedArray([-1, 4, -1]))   # [-1, -1, 4]

[-1, -1, 4]


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`**. Cả bốn cách cho cùng kết quả; Cách 3 ngắn nhất trong Python, Cách 4 là dạng "an toàn" khi chuyển sang ngôn ngữ có `%` giữ dấu.

## 8. TRACE từng bước

Trace **Cách 3** với `nums = [-1, 4, -1]` (`n = 3`):

| i | x = `nums[i]` | `i + x` | `(i + x) % 3` | `nums[...]` | `result` sau bước |
|---|---|---|---|---|---|
| 0 | -1 | -1 | 2 | -1 | `[-1]` |
| 1 | 4 | 5 | 2 | -1 | `[-1, -1]` |
| 2 | -1 | 1 | 1 | 4 | `[-1, -1, 4]` |

Chú ý hai dòng thú vị: `i = 0` (`-1 % 3 = 2`: đi trái quay vòng) và `i = 1` (`5 % 3 = 2`: đi phải quá một vòng, bước thừa bị loại).

Để thấy sự khác biệt với Cách 1, ta in cả **đường đi từng bước** của mô phỏng (hàm `trace_walk`) lẫn **kết quả công thức** (hàm `trace_formula`).

In [17]:
def trace_formula(nums):
    n = len(nums)
    result = []
    print(f"nums={nums}, n={n}")
    for i, x in enumerate(nums):
        raw = i + x
        land = raw % n
        result.append(nums[land])
        print(f"i={i} x={x:>3} i+x={raw:>3} (i+x)%n={land} nums[{land}]={nums[land]:>3} result={result}")
    return result

trace_formula([-1, 4, -1])
trace_formula([3, -2, 1, 1])

nums=[-1, 4, -1], n=3
i=0 x= -1 i+x= -1 (i+x)%n=2 nums[2]= -1 result=[-1]
i=1 x=  4 i+x=  5 (i+x)%n=2 nums[2]= -1 result=[-1, -1]
i=2 x= -1 i+x=  1 (i+x)%n=1 nums[1]=  4 result=[-1, -1, 4]
nums=[3, -2, 1, 1], n=4
i=0 x=  3 i+x=  3 (i+x)%n=3 nums[3]=  1 result=[1]
i=1 x= -2 i+x= -1 (i+x)%n=3 nums[3]=  1 result=[1, 1]
i=2 x=  1 i+x=  3 (i+x)%n=3 nums[3]=  1 result=[1, 1, 1]
i=3 x=  1 i+x=  4 (i+x)%n=0 nums[0]=  3 result=[1, 1, 1, 3]


[1, 1, 1, 3]

In [18]:
def trace_walk(nums, i):
    """In đường đi từng bước của phần tử i (Cách 1)."""
    n = len(nums)
    x = nums[i]
    pos = i
    path = [pos]
    steps = abs(x)
    for _ in range(steps):
        pos += 1 if x > 0 else -1
        if pos == n:
            pos = 0
        elif pos < 0:
            pos = n - 1
        path.append(pos)
    print(f"i={i}, x={x}: đường đi {path} -> đáp xuống {pos}, result[{i}] = {nums[pos]}")
    return pos

for i in range(4):
    trace_walk([3, -2, 1, 1], i)
print("---")
trace_walk([-1, 4, -1], 1)          # đi phải 4 bước, quay vòng hơn một vòng

i=0, x=3: đường đi [0, 1, 2, 3] -> đáp xuống 3, result[0] = 1
i=1, x=-2: đường đi [1, 0, 3] -> đáp xuống 3, result[1] = 1
i=2, x=1: đường đi [2, 3] -> đáp xuống 3, result[2] = 1
i=3, x=1: đường đi [3, 0] -> đáp xuống 0, result[3] = 3
---
i=1, x=4: đường đi [1, 2, 0, 1, 2] -> đáp xuống 2, result[1] = -1


2

## 9. Code nộp LeetCode

Ta nộp **Cách 3** viết với vòng `for` rõ ràng (dễ đọc hơn một dòng và tương đương). Trên LeetCode, `List` đã được import sẵn.

In [19]:
from typing import List

class Solution:
    def constructTransformedArray(self, nums: List[int]) -> List[int]:
        n = len(nums)
        result = []
        for i, x in enumerate(nums):
            land = (i + x) % n          # đi x bước trên vòng tròn (x âm = đi trái)
            result.append(nums[land])   # đọc từ mảng GỐC
        return result


for nums, expected in examples:
    got = Solution().constructTransformedArray(nums)
    print(nums, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[3, -2, 1, 1] -> [1, 1, 1, 3] OK
[-1, 4, -1] -> [-1, -1, 4] OK


## 10. Vì sao thuật toán đúng

**Cơ sở.** Đánh số các vị trí `0..n-1` quanh một vòng tròn. Đi **1 bước phải** từ `p` tới `(p + 1) mod n`; đi **1 bước trái** tới `(p - 1) mod n` (quy ước `mod` cho kết quả trong `0..n-1`). Đi `k` bước phải liên tiếp ⇒ `(p + k) mod n`; đi `k` bước trái ⇒ `(p - k) mod n`.

**Gộp hai hướng.** Với `nums[i] = x`: nếu `x > 0` đi `x` bước phải ⇒ `(i + x) mod n`. Nếu `x < 0` đi `|x| = -x` bước trái ⇒ `(i - |x|) mod n = (i + x) mod n`. Nếu `x = 0` ⇒ `(i + 0) mod n = i` và `nums[i] = 0 = x`. Cả ba trường hợp đều là `(i + x) mod n`, nên `result[i] = nums[(i + nums[i]) mod n]`.

**Python `%`.** Với `n > 0`, `a % n` luôn trong `0..n-1` và đồng dư với `a` theo modulo `n`, kể cả `a < 0`. Vậy chỉ số tìm được luôn hợp lệ và đúng.

**Tính độc lập.** Công thức chỉ **đọc** `nums`, và `result` là danh sách riêng, nên các `result[i]` không ảnh hưởng nhau — đúng yêu cầu "độc lập" của đề.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `M = max |nums[i]|`.

- **Cách 1 (đi từng bước):** Time `O(n * M)`, Space `O(n)`.
- **Cách 2 (tách hướng + `%`):** Time `O(n)`, Space `O(n)`.
- **Cách 3 (công thức gộp):** Time `O(n)`, Space `O(n)`.
- **Cách 4 (công thức an toàn):** Time `O(n)`, Space `O(n)`.

Space `O(n)` là cho mảng kết quả mà đề **yêu cầu trả về** (không tính là phụ). Thời gian `O(n)` là tối ưu vì phải điền `n` ô của kết quả.

## 12. Bộ test

### 12.1. Ví dụ của đề và các trường hợp biên

In [20]:
impls = {
    "walk":    transform_walk,
    "branch":  lambda a: SolutionBranch().constructTransformedArray(a),
    "formula": lambda a: SolutionFormula().constructTransformedArray(a),
    "safe":    lambda a: SolutionSafe().constructTransformedArray(a),
    "final":   lambda a: Solution().constructTransformedArray(a),
    "cstyle":  transform_cstyle,
}

cases = [
    ([3, -2, 1, 1], [1, 1, 1, 3]),      # ví dụ 1
    ([-1, 4, -1], [-1, -1, 4]),         # ví dụ 2
    ([0], [0]),                         # tự dựng: n = 1, đứng yên
    ([5], [5]),                         # tự dựng: n = 1, đi 5 bước vẫn về chính nó
    ([-100], [-100]),                   # tự dựng: n = 1, đi trái 100 bước
    ([0, 0, 0], [0, 0, 0]),             # tự dựng: toàn 0
    ([1, 2], [2, 2]),                   # tự dựng: i=0 -> nums[1]=2; i=1 đi phải 2 bước (về lại) -> nums[1]=2
    ([2, 2], [2, 2]),                   # tự dựng: đi 2 bước trên vòng 2 phần tử về lại chính nó
    ([1, 2, 3], [2, 1, 3]),             # tự dựng: tính tay: i0->nums[1]=2; i1->nums[0]=1; i2->nums[2]=3 ... xem bên dưới
    ([-1, -1, -1], [-1, -1, -1]),       # tự dựng: toàn -1
]

# kiểm tra ô cần tính tay: [1,2,3] n=3: i0:1 -> idx 1 -> 2; i1:2 -> idx 3%3=0 -> 1; i2:3 -> idx 5%3=2 -> 3
for nums, exp in cases:
    for name, f in impls.items():
        got = f(nums)
        assert got == exp, (name, nums, got, exp)
print("12.1 OK:", len(cases), "test x", len(impls), "cách")

12.1 OK: 10 test x 6 cách


Ghi chú test `[1, 2, 3]`: `n = 3`. `i=0` (`x=1`) → chỉ số `1` → `2`; `i=1` (`x=2`) → chỉ số `3 % 3 = 0` → `1`; `i=2` (`x=3`) → chỉ số `5 % 3 = 2` → `3`. Kết quả `[2, 1, 3]`, khớp.

### 12.2. Phản ví dụ cho các cách làm sai

Ta viết các bản sai có chủ đích rồi tìm input làm lộ lỗi.

In [21]:
def wrong_inplace(nums):
    """Sai: vừa đọc vừa ghi cùng một danh sách."""
    n = len(nums)
    res = nums                       # alias, không phải bản sao
    for i in range(n):
        res[i] = nums[(i + nums[i]) % n]
    return res

def wrong_no_wrap(nums):
    """Sai: quên quay vòng (chỉ dùng chỉ số i + x)."""
    out = []
    for i, x in enumerate(nums):
        out.append(nums[i + x])      # có thể IndexError hoặc chỉ số âm cho kết quả sai
    return out

def wrong_ignore_sign(nums):
    """Sai: luôn đi sang phải |x| bước."""
    n = len(nums)
    return [nums[(i + abs(x)) % n] for i, x in enumerate(nums)]

def wrong_mod_n_minus_1(nums):
    """Sai: quay vòng bằng len(nums) - 1."""
    n = len(nums)
    m = max(n - 1, 1)
    return [nums[(i + x) % m] for i, x in enumerate(nums)]

print("inplace       :", wrong_inplace([3, -2, 1, 1]), "(đúng: [1, 1, 1, 3])")
try:
    wrong_no_wrap([3, -2, 1, 1])
except IndexError as e:
    print("no_wrap       : IndexError:", e)
print("ignore_sign   :", wrong_ignore_sign([-1, 4, -1]), "(đúng: [-1, -1, 4])")
print("mod_n_minus_1 :", wrong_mod_n_minus_1([3, -2, 1, 1]), "(đúng: [1, 1, 1, 3])")

assert wrong_inplace([3, -2, 1, 1]) != [1, 1, 1, 3]
assert wrong_ignore_sign([-1, 4, -1]) != [-1, -1, 4]
assert wrong_mod_n_minus_1([3, -2, 1, 1]) != [1, 1, 1, 3]
print("12.2 OK: các bản sai đều bị lộ")

inplace       : [1, 1, 1, 1] (đúng: [1, 1, 1, 3])
no_wrap       : IndexError: list index out of range
ignore_sign   : [4, -1, -1] (đúng: [-1, -1, 4])
mod_n_minus_1 : [3, 1, 3, -2] (đúng: [1, 1, 1, 3])
12.2 OK: các bản sai đều bị lộ


### 12.3. Đối chiếu ngẫu nhiên (có seed) với bộ mô phỏng "đi từng bước"

Ta so các cách với **Cách 1 (đi từng bước)** — cách không dùng `%` nên **độc lập** với các công thức — trên nhiều input ngẫu nhiên đúng ràng buộc: `1 <= n <= 100`, `-100 <= nums[i] <= 100`. Thêm các trường hợp thiên về bước lớn so với `n` (quay vòng nhiều vòng).

In [22]:
import random

random.seed(3379)
count = 0
for _ in range(1200):
    n = random.randint(1, 8)
    lim = random.choice([2, 5, 100])
    nums = [random.randint(-lim, lim) for _ in range(n)]
    exp = transform_walk(nums)
    for name, f in impls.items():
        assert f(list(nums)) == exp, (name, nums)
    count += 1

for _ in range(100):
    n = random.randint(1, 100)
    nums = [random.randint(-100, 100) for _ in range(n)]
    exp = transform_walk(nums)
    for name, f in impls.items():
        assert f(list(nums)) == exp
    count += 1

# Đầu vào không bị sửa
a = [3, -2, 1, 1]
b = list(a)
Solution().constructTransformedArray(a)
assert a == b
print("12.3 OK:", count, "input ngẫu nhiên, các cách khớp mô phỏng; nums không bị sửa")

12.3 OK: 1300 input ngẫu nhiên, các cách khớp mô phỏng; nums không bị sửa


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Sửa `nums` tại chỗ khi vừa đọc vừa ghi

Nếu ta gán `nums[i] = nums[(i + nums[i]) % n]` ngay trong vòng lặp, các `i` sau sẽ đọc phải giá trị **đã bị thay đổi**, và cả `nums[i]` (số bước) cũng bị đổi. Đề yêu cầu mỗi phần tử xử lý **độc lập trên mảng gốc**.

In [23]:
# SAI
def sai_1(nums):
    n = len(nums)
    for i in range(n):
        nums[i] = nums[(i + nums[i]) % n]     # <-- ghi đè mảng gốc
    return nums

print(sai_1([3, -2, 1, 1]))        # [1, 1, 1, 1]  (sai; đúng là [1, 1, 1, 3])

# ĐÚNG: mảng kết quả riêng
print(Solution().constructTransformedArray([3, -2, 1, 1]))    # [1, 1, 1, 3]

[1, 1, 1, 1]
[1, 1, 1, 3]


### Lỗi 2 — Quên quay vòng

`nums[i + x]` ném `IndexError` khi `i + x >= n`. Với `i + x < 0` thì Python **âm chỉ số** vẫn "chạy" nhưng chỉ đúng khi `i + x >= -n`; bước lớn sẽ sai hoặc lỗi.

In [24]:
nums = [3, -2, 1, 1]
try:
    print(nums[0 + 3], nums[0 + 5])   # chỉ số 5 vượt biên
except IndexError as e:
    print("IndexError:", e)

nums = [5, 6, 7]
print(nums[-1])                       # 7: chỉ số âm nhỏ vô tình đúng
try:
    print(nums[-4])                   # đi trái quá xa -> lỗi
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range
7
IndexError: list index out of range


### Lỗi 3 — Bỏ qua dấu của `nums[i]`

Dùng `abs` rồi luôn đi sang phải làm mất hướng trái. Ví dụ `i = 0`, `x = -1` đi trái tới chỉ số cuối, còn đi phải 1 bước tới chỉ số `1`.

In [25]:
def sai_3(nums):
    n = len(nums)
    return [nums[(i + abs(x)) % n] for i, x in enumerate(nums)]

print(sai_3([-1, 4, -1]))          # [4, -1, -1]  (sai)
print(Solution().constructTransformedArray([-1, 4, -1]))   # [-1, -1, 4]

[4, -1, -1]
[-1, -1, 4]


### Lỗi 4 — Quay vòng bằng `len(nums) - 1` hoặc chỉ dùng một `if`

Mảng có `n` vị trí nên mô-đun là **`n`**, không phải `n - 1`. Và một lệnh `if idx >= n: idx -= n` chỉ đủ khi bước `< n`; bước lớn cần `%`.

In [26]:
def sai_4(nums):
    n = len(nums)
    out = []
    for i, x in enumerate(nums):
        idx = i + x
        if idx >= n:
            idx -= n                   # chỉ trừ một lần: không đủ khi x lớn
        if idx < 0:
            idx += n
        out.append(nums[idx])
    return out

# nums = [0, 0, 9], i = 2, x = 9: i + x = 11, trừ n một lần còn 8, vẫn vượt biên (n = 3)
try:
    print(sai_4([0, 0, 9]))
except IndexError as e:
    print("IndexError:", e)
print(Solution().constructTransformedArray([0, 0, 9]))    # [0, 0, 9]

IndexError: list index out of range
[0, 0, 9]


### Lỗi 5 — Dùng `(i + x) % n` trong C++/Java mà quên số âm

Trong C++/Java, `(i + x) % n` có thể **âm** khi `i + x < 0` (ví dụ `-1 % 4 = -1`), gây truy cập chỉ số âm. Công thức an toàn là `(i + x % n + n) % n` (Cách 4). Trong Python điều này **không** xảy ra, nhưng hãy nhớ khi chuyển ngôn ngữ.

In [27]:
print(c_mod(-1, 4), -1 % 4)            # -1 (kiểu C) và 3 (Python)
# Cách 4 đúng ngay cả khi % có thể âm
for i, x, n in [(0, -1, 4), (1, -2, 4), (2, -100, 3), (0, -3, 3)]:
    print(f"i={i} x={x} n={n}: an toàn = {(i + c_mod(x, n) + n) % n}, python = {(i + x) % n}")
    assert (i + c_mod(x, n) + n) % n == (i + x) % n

-1 3
i=0 x=-1 n=4: an toàn = 3, python = 3
i=1 x=-2 n=4: an toàn = 3, python = 3
i=2 x=-100 n=3: an toàn = 1, python = 1
i=0 x=-3 n=3: an toàn = 0, python = 0


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy **"mảng vòng / circular / wrap around"** ⇒ nghĩ ngay **phép `% n`**.
- **Đi `k` bước sang phải/trái** trên vòng tròn ⇒ `(i ± k) % n`; trong Python dấu trừ không cần xử lý thêm.
- "Mỗi phần tử xử lý **độc lập**" ⇒ dựng **mảng kết quả mới**, chỉ đọc mảng gốc.
- Các trường hợp đặc biệt trong đề (`nums[i] == 0`) thường **tự khớp** với công thức tổng quát; hãy thử công thức trước khi viết nhánh riêng.
- Khi nghi ngờ công thức, viết **bộ mô phỏng từng bước** (Cách 1) để đối chiếu — đúng cách ta làm ở mục 12.3.
- Ràng buộc nhỏ (`n <= 100`, `|x| <= 100`) cho phép cả mô phỏng thô; nhưng công thức `O(n)` gọn hơn và dễ chứng minh.
- Khi chuyển sang C++/Java, nhớ `%` giữ dấu: dùng `((a % n) + n) % n`.

## 15. Biến thể của bài

- **Trả về chỉ số đáp xuống** thay vì giá trị: `[(i + x) % n for i, x in enumerate(nums)]` (bài tập 4).
- **Áp dụng phép biến đổi nhiều lần (`t` lần):** lặp lại `t` lần; có thể xuất hiện chu kỳ (bài tập 5).
- **Đếm số `i` đáp xuống chính nó:** các `i` với `nums[i] % n == 0`.
- **Circular Array Loop (LeetCode 457):** bài khó hơn dùng cùng ý tưởng nhảy trên vòng tròn.
- **Rotate Array (LeetCode 189):** chuyển mọi phần tử `k` bước, dùng `(i + k) % n`.
- **Mảng vòng 2 chiều / lưới xuyến (torus):** `(r + dr) % R`, `(c + dc) % C`.
- **Đi mỗi lần `nums[i]` bước nhưng "bước kế tiếp" dựa trên vị trí mới:** biến thể "nhảy liên tục"; cần theo dõi chu kỳ.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đi từng bước | Với mỗi `i`, bước `\|x\|` lần, `if` quay vòng | `O(n * M)` | `O(n)` | Hiểu đề; làm oracle kiểm tra |
| 2. Tách hướng + `%` | `x > 0`: `(i+x)%n`; `x < 0`: `(i-\|x\|)%n` | `O(n)` | `O(n)` | Khi muốn code bám sát mô tả đề |
| 3. Công thức gộp | `nums[(i + x) % n]` | `O(n)` | `O(n)` | **Cách nộp chính** trong Python |
| 4. Công thức an toàn | `nums[(i + x % n + n) % n]` | `O(n)` | `O(n)` | Khi viết C++/Java (hoặc cẩn thận số âm) |

Mọi cách cùng kết quả (đã đối chiếu ở mục 12.3).

## 17. 5 bài tập vận dụng (dễ → khó)

Các cell TODO **không được gọi khi chạy notebook** — hãy tự điền rồi tự chạy. Đáp án ở mục 19.

### Bài 1 — Đoán output

Đoán kết quả `bai1()` (không chạy).

In [28]:
def bai1():
    nums = [2, -1, 0, 4]
    n = len(nums)
    return [nums[(i + x) % n] for i, x in enumerate(nums)]

# Hãy đoán rồi mới chạy:  print(bai1())

### Bài 2 — Tính tay

Với `nums = [1, -3, 2, -1, 5]` (`n = 5`): với từng `i`, viết `i + nums[i]`, chỉ số sau `% 5`, rồi giá trị. Điền kết quả cuối `result` vào biến dưới đây (không chạy hàm để lấy đáp án).

In [29]:
# TODO: tự tính tay rồi điền
bai2_result = None      # danh sách 5 phần tử

### Bài 3 — Điền TODO

Hoàn thành hàm theo Cách 2 (tách hai hướng): với `x > 0` dùng `(i + x) % n`, với `x < 0` dùng `(i - abs(x)) % n`, với `x == 0` giữ `i`.

In [30]:
def bai3(nums):
    n = len(nums)
    result = []
    for i in range(n):
        x = nums[i]
        # TODO: tính chỉ số land theo ba trường hợp
        land = ...
        result.append(nums[land])
    return result

### Bài 4 — Tự viết lại

Viết `bai4(nums)` trả về **danh sách chỉ số đáp xuống** (không phải giá trị). Ví dụ `bai4([3, -2, 1, 1]) = [3, 3, 3, 0]`, `bai4([-1, 4, -1]) = [2, 2, 1]`.

In [31]:
def bai4(nums):
    # TODO: tự viết
    ...

### Bài 5 — Biến thể / mini challenge

Viết `bai5(nums, t)` áp dụng phép biến đổi **`t` lần liên tiếp** (mỗi lần dùng mảng của lần trước làm `nums`), trả mảng cuối. Ví dụ `bai5([3, -2, 1, 1], 0) = [3, -2, 1, 1]`, `bai5([3, -2, 1, 1], 1) = [1, 1, 1, 3]`, `bai5([3, -2, 1, 1], 2) = [1, 1, 3, 1]`.

In [32]:
def bai5(nums, t):
    # TODO: tự viết (gợi ý: lặp t lần, mỗi lần tạo mảng mới)
    ...

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.**
- Mức 1: `n = 4`, tính `i + x` rồi `% 4` cho từng phần tử.
- Mức 2: `i=0: 0+2=2 → nums[2]=0`; `i=1: 1-1=0 → nums[0]=2`; `i=2: 2+0=2 → nums[2]=0`; `i=3: 3+4=7 → 7%4=3 → nums[3]=4`.
- Mức 3: kết quả `[0, 2, 0, 4]`.

**Bài 2.**
- Mức 1: `i + nums[i]` lần lượt `1, -2, 4, 2, 9`.
- Mức 2: sau `% 5`: `1, 3, 4, 2, 4` (vì `-2 % 5 = 3`, `9 % 5 = 4`).
- Mức 3: giá trị `nums[1] = -3`, `nums[3] = -1`, `nums[4] = 5`, `nums[2] = 2`, `nums[4] = 5`; kết quả `[-3, -1, 5, 2, 5]`.

**Bài 3.**
- Mức 1: ba nhánh `if x > 0 / elif x < 0 / else`.
- Mức 2: nhánh `x < 0`: `(i - abs(x)) % n`.
- Mức 3: `land = (i + x) % n if x > 0 else ((i - abs(x)) % n if x < 0 else i)`.

**Bài 4.**
- Mức 1: giống công thức nhưng không tra `nums[...]`.
- Mức 2: kết quả là chỉ số: `(i + x) % n`.
- Mức 3: `return [(i + x) % len(nums) for i, x in enumerate(nums)]`.

**Bài 5.**
- Mức 1: `t = 0` trả bản sao nguyên mảng.
- Mức 2: dùng vòng `for _ in range(t)`; mỗi lần `cur = [cur[(i + x) % n] for i, x in enumerate(cur)]`.
- Mức 3: nhớ tạo `cur` **mới** từ `cur` cũ (đọc mảng cũ, ghi mảng mới); trả `cur` cuối cùng.

## 19. Đáp án tham khảo cho bài tập

Mỗi đáp án đều được kiểm tra bằng `assert` (đối chiếu với mô phỏng đi từng bước).

In [33]:
# --- Bài 1 ---
assert bai1() == [0, 2, 0, 4]
assert transform_walk([2, -1, 0, 4]) == [0, 2, 0, 4]
print("Bài 1:", bai1())

# --- Bài 2 ---
bai2_result_da = [-3, -1, 5, 2, 5]
assert Solution().constructTransformedArray([1, -3, 2, -1, 5]) == bai2_result_da
assert transform_walk([1, -3, 2, -1, 5]) == bai2_result_da
print("Bài 2:", bai2_result_da)

# --- Bài 3 ---
def bai3_da(nums):
    n = len(nums)
    result = []
    for i in range(n):
        x = nums[i]
        if x > 0:
            land = (i + x) % n
        elif x < 0:
            land = (i - abs(x)) % n
        else:
            land = i
        result.append(nums[land])
    return result

random.seed(3380)
for _ in range(300):
    n = random.randint(1, 8)
    nums = [random.randint(-20, 20) for _ in range(n)]
    assert bai3_da(nums) == transform_walk(nums)
print("Bài 3 OK")

# --- Bài 4 ---
def bai4_da(nums):
    return [(i + x) % len(nums) for i, x in enumerate(nums)]

def walk_index(nums, i):
    n, pos, x = len(nums), i, nums[i]
    for _ in range(abs(x)):
        pos = (pos + (1 if x > 0 else -1)) % n
    return pos

assert bai4_da([3, -2, 1, 1]) == [3, 3, 3, 0]
assert bai4_da([-1, 4, -1]) == [2, 2, 1]
for _ in range(300):
    n = random.randint(1, 8)
    nums = [random.randint(-20, 20) for _ in range(n)]
    assert bai4_da(nums) == [walk_index(nums, i) for i in range(n)]
print("Bài 4 OK")

# --- Bài 5 ---
def bai5_da(nums, t):
    cur = list(nums)
    for _ in range(t):
        n = len(cur)
        cur = [cur[(i + x) % n] for i, x in enumerate(cur)]
    return cur

assert bai5_da([3, -2, 1, 1], 0) == [3, -2, 1, 1]
assert bai5_da([3, -2, 1, 1], 1) == [1, 1, 1, 3]
assert bai5_da([3, -2, 1, 1], 2) == [1, 1, 3, 1]
for _ in range(100):
    n = random.randint(1, 6)
    nums = [random.randint(-9, 9) for _ in range(n)]
    t = random.randint(0, 4)
    cur = list(nums)
    for _ in range(t):
        cur = transform_walk(cur)
    assert bai5_da(nums, t) == cur
print("Bài 5 OK")

Bài 1: [0, 2, 0, 4]
Bài 2: [-3, -1, 5, 2, 5]
Bài 3 OK
Bài 4 OK
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (công thức gộp):**

```python
class Solution:
    def constructTransformedArray(self, nums):
        n = len(nums)
        return [nums[(i + x) % n] for i, x in enumerate(nums)]
```

**Công thức cần nhớ:** `result[i] = nums[(i + nums[i]) % n]` — `%` của Python luôn cho `0..n-1` nên hướng trái tự xử lý.

**Checklist trước khi submit:**

- [ ] Tên method đúng: `constructTransformedArray(self, nums)` (không phải `transformedArray`).
- [ ] Trả về **mảng mới**, không sửa `nums` và không gán `result = nums`.
- [ ] Dùng `% n` (không phải `% (n - 1)`), kể cả khi bước lớn hơn `n`.
- [ ] Hướng trái là số âm: `(i + x) % n` đã bao hàm, không dùng `abs` rồi quên dấu.
- [ ] Nếu viết C++/Java: dùng `(i + x % n + n) % n`.
- [ ] Đã thử: `n = 1`, `x = 0`, `|x| > n`, `x` âm lớn, mảng toàn một giá trị.